In [ ]:
# Deepfake Detection Inference and Visualization
# This notebook loads a trained ResNext+LSTM model and predicts whether a 
# given video is REAL or FAKE. It also generates a heatmap to visualize 
# the areas the model focused on.

# Requirement: GPU should be enabled for faster inference.

In [ ]:
# Utilities for video processing and prediction visualization.
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import face_recognition
from torch import nn
from torchvision import models

In [ ]:
# Define the Model architecture (must be identical to the one used in training)
class Model(nn.Module):
    def __init__(self, num_classes, latent_dim=2048, lstm_layers=1, hidden_dim=2048, bidirectional=False):
        super(Model, self).__init__()
        # Use ResNext50 as the feature extractor CNN
        model = models.resnext50_32x4d(pretrained=True) # Load pre-trained ResNext
        # Extract all layers except the last two (pooling and fc)
        self.model = nn.Sequential(*list(model.children())[:-2]) # Remove task-specific layers
        # LSTM for processing the temporal sequence of features
        self.lstm = nn.LSTM(latent_dim, hidden_dim, lstm_layers, bidirectional) # Initialize LSTM
        self.relu = nn.LeakyReLU() # Non-linear activation
        self.dp = nn.Dropout(0.4) # Dropout for regularization
        self.linear1 = nn.Linear(2048, num_classes) # Final classification layer
        self.avgpool = nn.AdaptiveAvgPool2d(1) # Global average pooling

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.shape # Unpack input tensor shape
        # Flatten batch and sequence for CNN processing
        x = x.view(batch_size * seq_length, c, h, w) # Reshape for parallel CNN pass
        fmap = self.model(x) # Spatial feature maps from ResNext
        x = self.avgpool(fmap) # Apply pooling to reduce dimensionality
        x = x.view(batch_size, seq_length, 2048) # Reshape for LSTM sequence input
        x_lstm, _ = self.lstm(x, None) # Temporal feature extraction via LSTM
        # Use the last output of the LSTM for classification
        return fmap, self.dp(self.linear1(x_lstm[:, -1, :])) # Return maps and classification


In [ ]:
# Image normalization parameters matching the training step
im_size = 112
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
sm = nn.Softmax(dim=1) # Softmax to get probabilities

# Inverse normalization to convert tensors back to viewable images
inv_normalize = transforms.Normalize(mean=-1*np.divide(mean, std), std=np.divide([1, 1, 1], std))

def im_convert(tensor):
    """ Converts a PyTorch tensor back into a displayable NumPy image. """
    image = tensor.to("cpu").clone().detach() # Move to CPU and detach from graph
    image = image.squeeze() # Remove extra dimensions
    image = inv_normalize(image) # Reverse normalization
    image = image.numpy() # Convert to NumPy array
    image = image.transpose(1, 2, 0) # CHW to HWC for plotting
    image = image.clip(0, 1) # Ensure pixel values are within range
    return image # Return displayable image

def predict(model, img, path='./'):
  """ Performs prediction on an image sequence and generates a focus heatmap. """
  fmap, logits = model(img.to('cuda')) # Forward pass on GPU
  weight_softmax = model.linear1.weight.detach().cpu().numpy() # Extract layer weights
  logits = sm(logits) # Convert logits to probabilities
  _, prediction = torch.max(logits, 1) # Get the index of the highest probability
  confidence = logits[:, int(prediction.item())].item() * 100 # Calculate confidence percentage
  print('confidence of prediction:', confidence) # Print result
  
  # Heatmap generation (Grad-CAM style visualization)
  idx = np.argmax(logits.detach().cpu().numpy()) # Index of winning class
  bz, nc, h, w = fmap.shape # Shape of feature maps
  # Multiply feature maps by the weights of the winning class
  out = np.dot(fmap[-1].detach().cpu().numpy().reshape((nc, h*w)).T, weight_softmax[idx, :].T) # Compute activation
  predict = out.reshape(h, w) # Reshape to 2D
  # Normalize the heatmap
  predict = predict - np.min(predict) # Shift to positive range
  predict_img = predict / np.max(predict) # Scale to [0, 1]
  predict_img = np.uint8(255 * predict_img) # Scale to [0, 255] for OpenCV
  
  # Overlay heatmap onto the original image
  out = cv2.resize(predict_img, (im_size, im_size)) # Resize heatmap to match image
  heatmap = cv2.applyColorMap(out, cv2.COLORMAP_JET) # Apply color mapping
  img_orig = im_convert(img[:, -1, :, :, :]) # Convert the last frame for display
  result = heatmap * 0.5 + img_orig * 0.8 * 255 # Blend heatmap and original image
  
  plt.imshow(result.astype(np.uint8)) # Show blended image
  plt.title(f"Prediction: {'REAL' if prediction.item()==1 else 'FAKE'} ({confidence:.2f}%)") # Add title
  plt.show() # Display plot
  return [int(prediction.item()), confidence] # Return prediction results
#img = train_data[100][0].unsqueeze(0)
#predict(model,img)

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import face_recognition
class validation_dataset(Dataset):
    def __init__(self,video_names,sequence_length = 60,transform = None):
        self.video_names = video_names # List of video paths
        self.transform = transform # Transform pipeline
        self.count = sequence_length # Desired sequence length
    def __len__(self):
        return len(self.video_names) # Return total videos
    def __getitem__(self,idx):
        video_path = self.video_names[idx] # Video to process
        frames = [] # List to hold frames
        a = int(100/self.count) # Determine step size
        first_frame = np.random.randint(0,a) # Pick random starting point
        for i,frame in enumerate(self.frame_extract(video_path)): # Iterate over frames
            faces = face_recognition.face_locations(frame) # Detect faces
            try:
              top,right,bottom,left = faces[0] # Take the first detected face
              frame = frame[top:bottom,left:right,:] # Crop to face region
            except: # If no face detected
              pass # Use full frame
            frames.append(self.transform(frame)) # Apply transformations
            if(len(frames) == self.count): # Stop when sequence is full
              break
        frames = torch.stack(frames) # Stack frames into sequence
        frames = frames[:self.count] # Truncate if needed
        return frames.unsqueeze(0) # Return with batch dimension
    def frame_extract(self,path):
      vidObj = cv2.VideoCapture(path) # Open video
      success = 1 # Success flag
      while success: # Loop frames
          success, image = vidObj.read() # Read frame
          if success: # If successful
              yield image # Yield frame
def im_plot(tensor):
    image = tensor.cpu().numpy().transpose(1,2,0)
    b,g,r = cv2.split(image)
    image = cv2.merge((r,g,b))
    image = image*[0.22803, 0.22145, 0.216989] +  [0.43216, 0.394666, 0.37645]
    image = image*255.0
    plt.imshow(image.astype(int))
    plt.show()

In [ ]:
#Code for making prediction
im_size = 112
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
                                        transforms.ToPILImage(),
                                        transforms.Resize((im_size,im_size)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean,std)])
path_to_videos = ['/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Balanced_Face_only_data/aagfhgtpmv.mp4',
                                   '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Balanced_Face_only_data/aczrgyricp.mp4',
                                   '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Balanced_Face_only_data/agdkmztvby.mp4',
                                   '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Balanced_Face_only_data/abarnvbtwb.mp4']

path_to_videos = ['/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Youtube_Face_only_data/000_003.mp4',
                  '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Youtube_Face_only_data/000.mp4',
                  '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Youtube_Face_only_data/002_006.mp4',
                  '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Youtube_Face_only_data/002.mp4'
                  

]

path_to_videos= ["/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/DFDC_REAL_Face_only_data/aabqyygbaa.mp4"]

video_dataset = validation_dataset(path_to_videos,sequence_length = 20,transform = train_transforms)
model = Model(2).cuda()
path_to_model = '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Models/model_87_acc_20_frames_final_data.pt'
model.load_state_dict(torch.load(path_to_model))
model.eval()
for i in range(0,len(path_to_videos)):
  print(path_to_videos[i])
  prediction = predict(model,video_dataset[i],'./')
  if prediction[0] == 1:
    print("REAL")
  else:
    print("FAKE")

In [ ]:
#Optional : If you want to pass full frame for prediction instead of face cropped frame
#code for full frame processing
class validation_dataset(Dataset):
    def __init__(self,video_names,sequence_length = 60,transform = None):
        self.video_names = video_names
        self.transform = transform
        self.count = sequence_length
    def __len__(self):
        return len(self.video_names)
    def __getitem__(self,idx):
        video_path = self.video_names[idx]
        frames = []
        a = int(100/self.count)
        first_frame = np.random.randint(0,a) 
        for i,frame in enumerate(self.frame_extract(video_path)):
          frames.append(self.transform(frame))
          if(len(frames) == self.count):
            break
        frames = torch.stack(frames)
        frames = frames[:self.count]
        return frames.unsqueeze(0)
    def frame_extract(self,path):
      vidObj = cv2.VideoCapture(path) 
      success = 1
      while success:
          success, image = vidObj.read()
          if success:
              yield image